# Tema: Unity Catalog: catálogo, schema y objetos

## Objetivos
Usar nombres de tres niveles, tablas, vistas, volúmenes y privilegios.

## Conceptos importantes para el examen
Metastore → catálogo → schema → objeto. USE CATALOG y USE SCHEMA permiten resolver el espacio; SELECT lee; MODIFY escribe; CREATE TABLE crea tablas; CREATE SCHEMA crea schemas.

**Dificultad:** Intermedio · **Tiempo estimado:** 65 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_10_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
employees = spark.createDataFrame(
    [(i, f"Empleado {i:02d}", ["Data", "Sales", "Finance"][i % 3],
      30000 + i * 1500, i % 4 != 0,
      datetime(2026, 1, 1), datetime(2026, 1, 1)) for i in range(1, 19)],
    "employee_id INT, name STRING, department STRING, salary INT, active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP"
)
employees.createOrReplaceTempView("employees_seed")
employees.write.format("delta").mode("overwrite").saveAsTable("employees")
display(employees.orderBy("employee_id"))

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Contexto y nombres completos

In [ ]:
display(spark.sql("SELECT current_catalog(), current_schema(), current_user()"))
display(spark.sql(f"SELECT * FROM {ident(CATALOG)}.{ident(SCHEMA)}.employees LIMIT 3"))

### 2. Vista gobernada

In [ ]:
%sql
CREATE OR REPLACE VIEW active_employees AS SELECT employee_id, name, department FROM employees WHERE active;
SHOW TABLES;
SHOW GRANTS ON TABLE employees;

### 3. Volumen para archivos

In [ ]:
# Requiere CREATE VOLUME en el schema; alternativa: usa un volumen autorizado.
spark.sql("CREATE VOLUME IF NOT EXISTS lab_files")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/lab_files"
dbutils.fs.mkdirs(BASE + "/landing")
CHECKPOINT = BASE + "/checkpoints/main"
print(BASE)
dbutils.fs.put(BASE + "/readme.txt", "Datos ficticios de prácticas", overwrite=True)
display(dbutils.fs.ls(BASE))

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Crea una tabla uc_practice y consulta usando catalog.schema.table.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Crea una vista con solo employee_id y department y muéstrala.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Prepara CREATE CATALOG y CREATE SCHEMA. Si tienes CREATE CATALOG, activa su ejecución; en caso contrario demuestra el schema actual como alternativa.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Construye GRANT de lectura para un grupo existente y REVOKE de SELECT; ejecuta solo con autoridad para conceder permisos.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Comprueba que un volumen almacena archivos y una tabla se consulta con SELECT. Crea un JSON en el volumen y léelo.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** Construye identificadores escapados.

**Pista 2:** Una vista persiste la definición de consulta.

**Pista 3:** Crear catálogo necesita privilegio en metastore y almacenamiento configurado.

**Pista 4:** Incluye USE CATALOG, USE SCHEMA y SELECT; sustituye el grupo.

**Pista 5:** READ/WRITE VOLUME son permisos diferentes a SELECT/MODIFY.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
spark.sql("CREATE OR REPLACE TABLE uc_practice USING DELTA AS SELECT * FROM employees")
display(spark.sql(f"SELECT COUNT(*) FROM {ident(CATALOG)}.{ident(SCHEMA)}.uc_practice"))

### Solución 2

In [ ]:
%sql
CREATE OR REPLACE VIEW departments_public AS SELECT employee_id, department FROM employees;
SELECT * FROM departments_public;

### Solución 3

In [ ]:
RUN_CREATE_CATALOG = False
new_catalog = "dea_catalog_" + RUN_ID
commands = [f"CREATE CATALOG {ident(new_catalog)}", f"CREATE SCHEMA {ident(new_catalog)}.practice"]
for command in commands:
    print(command)
    if RUN_CREATE_CATALOG:
        spark.sql(command)
# Alternativa real ya ejecutada: CREATE SCHEMA en CATALOG autorizado.
display(spark.sql(f"SHOW SCHEMAS IN {ident(CATALOG)}"))

### Solución 4

In [ ]:
GROUP = "grupo_practicas"  # Grupo existente; no se crea automáticamente.
RUN_GRANTS = False
commands = [
 f"GRANT USE CATALOG ON CATALOG {ident(CATALOG)} TO {ident(GROUP)}",
 f"GRANT USE SCHEMA ON SCHEMA {ident(CATALOG)}.{ident(SCHEMA)} TO {ident(GROUP)}",
 f"GRANT SELECT ON TABLE employees TO {ident(GROUP)}",
 "SHOW GRANTS ON TABLE employees",
 f"REVOKE SELECT ON TABLE employees FROM {ident(GROUP)}"]
for command in commands:
    print(command)
    if RUN_GRANTS:
        display(spark.sql(command))
# Sin autoridad: SHOW GRANTS sobre la tabla propia es la práctica alternativa.

### Solución 5

In [ ]:
dbutils.fs.put(BASE + "/one.json", '{"id":1,"kind":"practice"}', overwrite=True)
display(spark.read.json(BASE + "/one.json"))
display(spark.sql("SELECT COUNT(*) FROM employees"))

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
SELECT está concedido, pero falta USE SCHEMA. ¿Qué debe revisarse?

A. El formato Parquet

B. Los privilegios de los contenedores

C. El nombre del driver

D. El checkpoint

### Pregunta 2
¿Qué privilegio permite escribir datos existentes de una tabla?

A. USE CATALOG

B. BROWSE

C. MODIFY

D. CREATE SCHEMA

### Pregunta 3
¿Qué almacena un volumen?

A. Archivos gobernados por UC

B. Solo vistas SQL

C. Únicamente secretos

D. Un scheduler

### Respuestas y explicación
**1. B** — El acceso a un objeto también requiere uso de sus contenedores.

**2. C** — MODIFY permite operaciones de modificación de datos.

**3. A** — Es un objeto gobernado para archivos no registrados como tablas.

## PARTE 6 - RETO FINAL
Organiza un schema con una tabla interna, una vista de consumo y un volumen de entrada. Define permisos mínimos para productor y lector y verifica lo que permita tu identidad.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
